# The second model — the last untested lever

```
0.899  submitted        rank ~1297/2792
0.926  bronze           0.944  gold
```

`notes/40` closed the config direction: ILP weights, repair chain, detection threshold, gap
span and short-track pruning are all located, several confirmed on two independent grids.
`notes/34` measured the deepcenter model at a ~0.002 ceiling. `notes/38` measured
bidirectional linking at +0.0036, and it needs no second model at all.

**This is the only remaining lever that adds model capacity** rather than re-reading the one
we have: `pilkwang/biohub-temporal-unet3d-seed314159-v1`, a second edge predictor of the
same architecture from a different seed. The public notebooks run it; we never have.

## How it is wired

The pack ships no ensemble hook, so `pipeline.secondary.patch_source` edits
`predict_video`'s source at two sites — the `model.encode` call and the `predict_edges`
call. **Both anchors were read from the pack's actual script, not guessed**, and
`probes/exec_secondary.py` applies the patch to that real source and compiles it. The
bidirectional build cost three launches to anchor mismatches and missing imports; this one
was checked on the ground truth first.

The secondary's logits are calibrated onto the primary's per-target mean and standard
deviation before mixing — the downstream candidate threshold and the ILP were both tuned
against the primary's scale. Two mixing modes:

* **`fixed`** — constant weight everywhere.
* **`low_margin`** — weight scales with the primary's own top-2 uncertainty and is **zeroed
  where the two models disagree** about the best parent. Where the primary is confident
  there is nothing to gain; where they disagree outright, averaging two contradictory
  answers is worse than either.

Everything else is held at the located configuration: det 0.975, gap 2, min-track 6, ILP
0.4/2.0, bidirectional w=0.15. **This run varies one thing.**

## Pre-registered predictions

1. **`w=0` reproduces `notes/40`'s 0.9535 (± 0.002)** on the same 12 datasets.
   `secondary_blend` returns the primary *by identity* at w=0, so a miss means the patch
   changed something it should not have, and nothing below is readable.
2. **The secondary actually fires** — candidate counts differ from the control by a real
   margin at some weight. A patch that matched but did nothing would present copies of the
   control as a sweep, which is exactly what `notes/38`'s prediction 2 was written to catch.
3. **`low_margin` beats `fixed` at matched weight.** The mode claim. If it does not, the
   gating is complexity for nothing and `fixed` is the honest choice.
4. **The best arm beats the control by more than 0.001** — the outcome claim at `notes/34`'s
   noise floor.
5. **The optimum is not at the largest weight tried.** Asked up front; three ILP sweeps each
   had to ask it again.

*`notes/40`: every mechanism this session has returned +0.002 to +0.004 on the leaderboard,
five for five in direction, none larger. Bronze needs +0.027. This is the last identified
lever, and it would have to be several times larger than anything measured so far to close
that — which is stated here so the result is read against a real expectation.*

In [ ]:
import os, subprocess, sys, time, json
from pathlib import Path

T_START = time.time()
WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)

def sh(*a, **kw):
    try:
        return subprocess.run(a, capture_output=True, text=True, **kw)
    except (FileNotFoundError, OSError) as e:
        return subprocess.CompletedProcess(a, 127, "", str(e))

def pip_install(pkgs, extra=()):
    r = sh(sys.executable, "-m", "pip", "install", "-q", *extra, *pkgs)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
    return r.returncode == 0

print(sh("nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader").stdout.strip()
      or "no GPU")

def find_dir(is_match, roots, max_depth=6):
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        stack = [(root, 0)]
        while stack:
            d, depth = stack.pop(0)
            try:
                if is_match(d):
                    return d
                if depth >= max_depth:
                    continue
                kids = [e for e in d.iterdir()
                        if e.is_dir() and e.suffix not in (".zarr", ".geff")]
            except (PermissionError, OSError):
                continue
            stack += [(k, depth + 1) for k in kids]
    return None

PACK = find_dir(lambda p: (p / "repo").is_dir() and (p / "weights").is_dir()
                and (p / "wheels").is_dir() and "seed314159" not in str(p),
                ["/kaggle/input"])
REPO = find_dir(lambda p: (p / "harness").is_dir() and (p / "pipeline").is_dir(),
                [WORK, "/kaggle/input"])
COMP = find_dir(lambda p: (p / "train").is_dir() and (p / "test").is_dir()
                and any((p / "train").glob("*.zarr")), ["/kaggle/input"])
# The torch wheelhouse: a directory of .whl files that is NOT the pack's own.
TORCH_WH = find_dir(
    lambda p: p.name == "wheels" and any(x.name.startswith("torch-") for x in p.iterdir()),
    ["/kaggle/input"])

for label, val in (("pack", PACK), ("our repo", REPO), ("competition", COMP),
                   ("torch wheels", TORCH_WH)):
    print(f"  {label:<14} {val}")
missing = [l for l, v in (("pack", PACK), ("our repo", REPO), ("competition", COMP)) if v is None]
if missing:
    raise SystemExit(f"not mounted: {missing}")
TRAIN = COMP / "train"
# Only to reuse notes/35's dataset list so the control is comparable; the candidates
# themselves are re-predicted here, not read from it.
CACHE = find_dir(lambda p: any(p.glob("cand_*.npz")), ["/kaggle/input"])
print(f"  {'cand cache':<14} {CACHE}")
# The ILP at 0.4/2.0 emits forks BY DESIGN (notes/35: div_J 0.1154), and purescore is
# only exact without them, so Harness.score_graph requires the official scorer.
CELLMOT = Path("/kaggle/working/kaggle-cell-tracking-competition")
if not (CELLMOT / "src" / "tracking_cellmot").is_dir():
    _r = sh("git", "clone", "--depth", "1",
            "https://github.com/royerlab/kaggle-cell-tracking-competition", str(CELLMOT))
    print(f"official scorer clone rc={_r.returncode}")
os.environ["CELLMOT_REPO"] = str(CELLMOT)
if not (CELLMOT / "src" / "tracking_cellmot").is_dir():
    raise SystemExit("official scorer not available; forked predictions cannot be scored")

# Offline installs, pack wheels first so numpy lands before anything compiles against it.
t0 = time.time()
ok1 = pip_install([str(p) for p in sorted((PACK / "wheels").glob("*.whl"))],
                  extra=("--no-index", f"--find-links={PACK/'wheels'}"))
print(f"pack wheels {'ok' if ok1 else 'FAILED'} ({time.time()-t0:.0f}s)")

if TORCH_WH is None:
    print("!! no torch wheelhouse attached — the P100 cannot run the image torch, so this "
          "will fall back to CPU and will NOT finish inside 12 h.")
else:
    t0 = time.time()
    ok2 = pip_install(["torch==2.5.1"], extra=("--no-index", f"--find-links={TORCH_WH}"))
    print(f"torch wheels {'ok' if ok2 else 'FAILED'} ({time.time()-t0:.0f}s)")

probe = sh(sys.executable, "-c",
           "import numpy, torch, zarr, tracksdata; "
           "ok=False\n"
           "if torch.cuda.is_available():\n"
           "    try:\n"
           "        w=torch.nn.Conv3d(1,4,3,padding=1).cuda()\n"
           "        _=w(torch.randn(2,1,8,8,8,device='cuda')).sum().item()\n"
           "        torch.cuda.synchronize(); ok=True\n"
           "    except Exception as e: print('GPU BROKEN:', type(e).__name__, str(e)[:120])\n"
           "print('numpy', numpy.__version__, '| torch', torch.__version__, '| gpu_ok', ok)")
print(probe.stdout.strip() or probe.stderr.strip()[-1500:])
if probe.returncode != 0:
    raise SystemExit("dependency stack does not import in a fresh interpreter — a scored "
                     "rerun would fail identically with no way to recover.")
if "gpu_ok True" not in probe.stdout:
    print("\n!! GPU is not usable. Continuing, but expect this to exceed the time budget; "
          "the guard below will still emit a valid file.")

## 1. Patch `predict_video`, then run the arms

`inspect.getsource` on the pack's own function, `patch_source` to insert the reverse pass,
`exec` into a **copy** of the pack module's namespace. The copy matters: mutating the pack
in place would make the control arm depend on run order.

In [ ]:
import subprocess, sys, time
WORKER = WORK / "run_bidir.py"
WORKER.write_text('''
import json, os, sys, time
from pathlib import Path
import numpy as np

os.environ["CELLMOT_REPO"] = {cellmot!r}
PACK = Path({pack!r}); REPO = Path({repo!r}); TRAIN = Path({train!r})
CACHE = {cache!r}; WORK = Path({work!r})
N_DATASETS = 12
BLEND_W = 0.15
DET_GRID = [0.975]
SEC_GRID = [tuple(g) for g in [(0.0, 'fixed'), (0.15, 'low_margin'), (0.3, 'low_margin'), (0.15, 'fixed'), (0.3, 'fixed'), (0.5, 'fixed')]]
POST_GRID = [tuple(g) for g in [(6, 2)]]
T0 = time.time()

sys.path.insert(0, str(REPO))
sys.path.insert(0, str(PACK / "repo" / "src"))
sys.path.insert(0, str(PACK / "repo" / "scripts"))
# The pack's entry point is a SCRIPT, not a package, and it imports a `dataspec` module
# that only exists in the authors' own environment. claude_submit_ratio injects a synthetic
# one; copied from there rather than retyped, which is how v1 of this notebook came to
# import a module name that does not exist.
import types
_ds = types.ModuleType("dataspec")
_ds.USERNAME = "claude"; _ds.INTERACTIVE = False
_ds.WEIGHTS_PATH = PACK / "weights"; _ds.DATASET_PATH = TRAIN
_ds.PREDICTIONS_PATH = WORK / "predictions"
sys.modules["dataspec"] = _ds

import inspect
import torch
import tracksdata as td
from harness import Harness
from harness.tracks import Tracks, read_geff, read_scale
from harness.purescore import summarise
from pipeline.anatomy import BUCKETS, edge_anatomy, summarise_anatomy
from pipeline.repair import close_gaps, linefit_smooth, prune_short_tracks
from pipeline.bidirectional import ANCHOR, harmonic_blend, patch_source
import predict_unet_transformer as P
print("worker numpy", np.__version__, "torch", torch.__version__, flush=True)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
ILP_EDGE_W, ILP_APP_W, ILP_DIS_W, ILP_DIV_W = -1.0, 0.4, 2.0, 1.0   # notes/36: the optimum
DET_THRESHOLD = 0.99

def repair_chain(g, sc):
    # A docstring here would terminate the outer f-string that writes this file.
    r = close_gaps(*g, scale=sc, max_um=5.75, max_added_frac=0.038, max_added_abs=1650)
    return linefit_smooth(*r, window=2, weight=0.76, scale=sc, max_shift_um=3.2)

ORIG_SRC = inspect.getsource(P.predict_video)
if ANCHOR not in ORIG_SRC:
    # Print the neighbourhood so the anchor can be fixed in ONE round trip rather than by
    # guessing. patch_source would catch it, but only after a GPU has been spent.
    i = ORIG_SRC.find("predict_edges")
    print("!! ANCHOR DOES NOT MATCH. predict_video source near predict_edges:", flush=True)
    print(ORIG_SRC[max(0, i - 500):i + 1000], flush=True)
    raise SystemExit("anchor mismatch -- update pipeline/bidirectional.ANCHOR")
print("anchor matches", ORIG_SRC.count(ANCHOR), "x in", len(ORIG_SRC), "chars", flush=True)

def make_predict(weight):
    # w=0 uses the ORIGINAL function object, so the control cannot differ from the
    # unpatched pipeline by even a rounding step.
    if weight == 0.0:
        return P.predict_video
    ns = dict(P.__dict__)
    exec(compile(patch_source(ORIG_SRC, weight), "<bidir>", "exec"), ns)
    return ns["predict_video"]

WPATH = PACK / "weights/unet_transformer/split_0/edge_predictor_best.pth"
model, window_size, downsample = P.load_model(WPATH, DEV)
print("model params", sum(p.numel() for p in model.parameters()),
      "window", window_size, "downsample", downsample, flush=True)
cfg = P.PredictConfig(det_threshold=DET_THRESHOLD, use_ilp=True,
                      ilp_edge_weight=ILP_EDGE_W, ilp_appearance_weight=ILP_APP_W,
                      ilp_disappearance_weight=ILP_DIS_W, ilp_division_weight=ILP_DIV_W)

# The same datasets notes/35 measured, so w=0 is comparable to 0.9179. The fallback is
# stratified -- notes/34's lesson: names[:12] sorted alphabetically gave 10 44b6 and 2
# 6bba, inverting a 71/128 population split.
names = []
if CACHE:
    names = sorted(p.stem[5:] for p in Path(CACHE).glob("cand_*.npz"))
    names = [n for n in names if (TRAIN / (n + ".geff")).exists()]
if not names:
    alln = sorted(p.stem for p in TRAIN.glob("*.zarr")
                  if (TRAIN / (p.stem + ".geff")).exists())
    a = [n for n in alln if n.startswith("44b6")]
    b = [n for n in alln if n.startswith("6bba")]
    k = max(1, round(N_DATASETS * len(a) / max(len(a) + len(b), 1)))
    names = a[:k] + b[:N_DATASETS - k]
# Stratify whatever list we ended up with. notes/34 recorded that names[:12] taken
# alphabetically inverted the embryo split; v2 of THIS notebook did it again, because the
# stratified branch only ran when the cache was missing. Slice proportionally, always.
_a = [n for n in names if n.startswith("44b6")]
_b = [n for n in names if not n.startswith("44b6")]
if _a and _b and len(names) > N_DATASETS:
    _k = max(1, round(N_DATASETS * len(_a) / len(names)))
    _k = min(_k, len(_a), N_DATASETS - 1)
    names = _a[:_k] + _b[:N_DATASETS - _k]
names = names[:N_DATASETS]
n44 = sum(n.startswith("44b6") for n in names)
print(len(names), "datasets:", n44, "x 44b6,", len(names) - n44, "x 6bba", flush=True)

h = Harness(data_dir=TRAIN, cache_dir=None)

def repair_at(g, sc, min_len, gap_max):
    # The submitted chain, with the two audited knobs exposed. min_len=0 / gap_max=1 is
    # byte-identical to what scored 0.897 -- probes/exec_config.py pins that on 30 random
    # graphs, so the anchor cell really is the current submission.
    r = close_gaps(*g, scale=sc, max_um=5.75, max_added_frac=0.038,
                   max_added_abs=1650, max_gap=gap_max)
    r = linefit_smooth(*r, window=2, weight=0.76, scale=sc, max_shift_um=3.2)
    if min_len > 0:
        r = prune_short_tracks(*r, min_frames=min_len, keep_division_components=True)
    return r


SEC_PATH = None
for _root in sorted(Path("/kaggle/input").rglob("*seed314159*")):
    _c = sorted(list(_root.rglob("*.pth")) + list(_root.rglob("*.pt"))) if _root.is_dir() \
        else ([_root] if _root.suffix in (".pth", ".pt") else [])
    if _c:
        SEC_PATH = _c[0]
        break
print("secondary weights:", SEC_PATH, flush=True)
if SEC_PATH is None:
    raise SystemExit("secondary model not mounted — attach "
                     "pilkwang/biohub-temporal-unet3d-seed314159-v1")

# The load-bearing check, and the reason v1 is being rerun. Both datasets ship the same
# directory layout, so a mis-resolved PACK silently makes primary and secondary the SAME
# FILE: the blend then mixes a model with itself, every arm equals the control, and the
# run reports "no gain" for a mechanism it never actually tested. Comparing the resolved
# paths is not enough -- two paths can point at identical bytes -- so compare the bytes.
import hashlib
def _digest(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        while (b := fh.read(1 << 20)):
            h.update(b)
    return h.hexdigest()
_dp, _ds_ = _digest(WPATH), _digest(SEC_PATH)
# Concatenation, not an f-string: this worker is rendered with .format(), so a literal
# brace here is eaten as a template field. The surrounding worker code avoids braces for
# the same reason, and the static check in the dry run is what caught this.
print("  primary   " + str(WPATH), flush=True)
print("            " + _dp[:16] + "  " + str(WPATH.stat().st_size) + " B", flush=True)
print("  secondary " + str(SEC_PATH), flush=True)
print("            " + _ds_[:16] + "  " + str(SEC_PATH.stat().st_size) + " B", flush=True)
if _dp == _ds_:
    raise SystemExit(
        "primary and secondary are the SAME WEIGHTS. An ensemble of a model with itself "
        "returns the control at every weight and would be reported as 'the mechanism does "
        "not pay'. Check which dataset resolved as PACK.")
print("two distinct models confirmed", flush=True)

SEC_MODEL, sec_w, sec_d = P.load_model(SEC_PATH, DEV)
if (sec_w, sec_d) != (window_size, downsample):
    # Different inference grids means the two models index different feature maps, and the
    # blend would pair unrelated cells while still producing plausible numbers.
    raise SystemExit("primary/secondary inference grids differ: "
                     + str((window_size, downsample)) + " vs " + str((sec_w, sec_d)))
print("secondary loaded, grids match", flush=True)

from pipeline.secondary import patch_source as sec_patch

def make_sec_predict(w, mode):
    # w=0 uses the ORIGINAL bidirectional-only function, so the control cannot differ from
    # the shipped pipeline by even a rounding step.
    base = make_predict(BLEND_W)
    if w == 0.0:
        return base, None
    ns = dict(P.__dict__)
    src = sec_patch(ORIG_SRC, w, mode=mode)
    from pipeline.bidirectional import patch_source as bid_patch
    src = bid_patch(src, BLEND_W)
    exec(compile(src, "<sec>", "exec"), ns)
    ns["_SECONDARY_MODEL"] = SEC_MODEL
    return ns["predict_video"], ns

LABELS = ["w" + str(w) + "_" + m for w, m in SEC_GRID]
ROWS = dict((l, []) for l in LABELS); ANAT = dict((l, []) for l in LABELS)
NODES = dict((l, 0) for l in LABELS); EDGES = dict((l, 0) for l in LABELS)
CAND = dict((l, 0) for l in LABELS); PER = {{}}
DET = DET_GRID[0]
MIN_LEN, GAP_MAX = POST_GRID[0]
cfg_d = P.PredictConfig(det_threshold=DET, use_ilp=True,
                        ilp_edge_weight=ILP_EDGE_W, ilp_appearance_weight=ILP_APP_W,
                        ilp_disappearance_weight=ILP_DIS_W, ilp_division_weight=ILP_DIV_W)
print("det", DET, "| min_len", MIN_LEN, "| gap", GAP_MAX, "|", len(SEC_GRID), "arms", flush=True)

for name in names:
    t0 = time.time()
    sc = read_scale(TRAIN / (name + ".zarr"))
    gt = read_geff(TRAIN / (name + ".geff"))
    parts = [name]
    for w, mode in SEC_GRID:
        lbl = "w" + str(w) + "_" + mode
        pv_s, _ns = make_sec_predict(w, mode)
        coords, edges = pv_s(model, TRAIN / (name + ".zarr"), DEV, cfg=cfg_d,
                             window_size=window_size, unet_batch_size=8,
                             downsample=downsample)
        g_td = P.build_graph(coords, edges)
        CAND[lbl] += int(g_td.num_edges())
        if g_td.num_edges():
            solver = td.solvers.ILPSolver(
                edge_weight=ILP_EDGE_W * td.EdgeAttr("edge_prob"),
                appearance_weight=ILP_APP_W, disappearance_weight=ILP_DIS_W,
                division_weight=ILP_DIV_W)
            with P.suppress_output():
                g_td = solver.solve(g_td)
        tr = Tracks.from_tracksdata(g_td)
        g = repair_at((tr.t, tr.zyx, tr.edges), sc, MIN_LEN, GAP_MAX)
        ROWS[lbl].append(h.score_graph(name, Tracks(g[0], g[1], g[2])))
        NODES[lbl] += int(len(g[0])); EDGES[lbl] += int(len(g[2]))
        a = edge_anatomy(g[0], g[1], g[2], gt.t, gt.zyx, gt.edges, scale=sc)
        ANAT[lbl].append(a)
        if sum(a[k] for k in BUCKETS) != a["n_gt_edges"]:
            raise SystemExit(name + "/" + lbl + ": buckets do not sum")
        PER.setdefault(name, {{}})[lbl] = float(
            ROWS[lbl][-1].get("adj_edge_jaccard", float("nan")))
        parts.append(lbl + " " + format(PER[name][lbl], ".4f"))
    print("  " + "  ".join(parts) + "   " + str(int(time.time() - t0)) + "s", flush=True)

    out = {{"arms": LABELS, "sec_grid": [list(g) for g in SEC_GRID], "det": DET,
           "post": [MIN_LEN, GAP_MAX], "blend_w": BLEND_W,
           "datasets": [n for n in names if n in PER],
           "summary": dict((l, summarise(ROWS[l])) for l in LABELS if ROWS[l]),
           "anatomy": dict((l, summarise_anatomy(ANAT[l])) for l in LABELS if ANAT[l]),
           "nodes": NODES, "edges": EDGES, "candidates": CAND, "per_dataset": PER}}
    (WORK / "secondary.json").write_text(json.dumps(out, indent=2, default=float))

print("worker done in", int(time.time() - T0), "s", flush=True)
'''.format(
    pack=str(PACK), repo=str(REPO), train=str(TRAIN), cache=str(CACHE or ""),
    work=str(WORK), cellmot=str(CELLMOT)))

t0 = time.time()
proc = subprocess.Popen([sys.executable, "-u", str(WORKER)],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line.rstrip(), flush=True)
rc = proc.wait()
print("worker exited", rc, "after", int(time.time() - t0), "s")
if rc != 0:
    raise SystemExit("worker failed (" + str(rc) + ")")

## 2. The five predictions

In [ ]:
import numpy as np, json
D = json.loads((WORK / "secondary.json").read_text())
S, A, N, E, C = D["summary"], D["anatomy"], D["nodes"], D["edges"], D["candidates"]
ARMS, DS, GRID = D["arms"], D["datasets"], [tuple(g) for g in D["sec_grid"]]
CTRL = "w0.0_fixed"
N40 = 0.9535                       # notes/40's located cell, on THESE 12 datasets
EXACT = CTRL in S and S[CTRL]["score"] == S[CTRL]["score"]
key = "score" if EXACT else "edge_jaccard"
print(f"{len(DS)} datasets, {len(ARMS)} arms  |  det {D['det']}  post {D['post']}  "
      f"blend {D['blend_w']}")
if not EXACT:
    print("!! score column is NaN (unreadable node budget) — grading on `edge_jaccard`.")
print()

print(f"{'arm':<18}{'score':>9}{'vs ctl':>9}{'edge_J':>9}{'div_J':>8}"
      f"{'mislink':>9}{'gap':>7}{'detect':>8}{'cand':>11}")
print("-" * 88)
for a in ARMS:
    if a not in S:
        continue
    st, an = S[a], A[a]
    dj = st.get("division_jaccard")
    print(f"{a:<18}{st[key]:>9.4f}{st[key]-S[CTRL][key]:>+9.4f}{st['edge_jaccard']:>9.4f}"
          f"{(dj if dj == dj else 0):>8.4f}{an['fn_mislink']:>9,}{an['fn_gap']:>7,}"
          f"{an['fn_detect']:>8,}{C.get(a, 0):>11,}")

print()
print("=" * 88)
print("PREDICTION GRADING")
print("=" * 88)

print("\n1. w=0 reproduces notes/40's 0.9535 (+-0.002)")
if not EXACT:
    print("   NOT GRADED — score column is NaN.")
else:
    got = S[CTRL]["score"]
    ok1 = abs(got - N40) <= 0.002
    print(f"   {CTRL} = {got:.4f} vs {N40:.4f}  ->  {'PASS' if ok1 else 'FAIL'}")
    if not ok1:
        print("   secondary_blend returns the primary BY IDENTITY at w=0 and the control")
        print("   uses the unpatched-for-secondary function, so a miss means the patch")
        print("   changed something it should not have. Nothing below is readable.")

print("\n2. the secondary actually fires")
c0 = C.get(CTRL, 0)
ok2 = False
for a in ARMS:
    if a == CTRL:
        continue
    d = (C.get(a, 0) - c0) / max(c0, 1)
    fired = abs(d) > 0.001
    ok2 |= fired
    print(f"   {a:<18} candidates {C.get(a,0):>9,}  vs {c0:,}  ({d:+.2%})"
          f"  {'ok' if fired else '<-- IDENTICAL'}")
print(f"   ->  {'PASS' if ok2 else 'FAIL'}")
if not ok2:
    print("   Every arm is a copy of the control: the patch matched but did nothing.")
    print("   Check that _SECONDARY_MODEL reached the patched function's globals.")

print("\n3. low_margin beats fixed at matched weight")
if not EXACT:
    print("   NOT GRADED — score column is NaN.")
else:
    pairs = [(w, f"w{w}_low_margin", f"w{w}_fixed") for w, m in GRID
             if m == "low_margin" and f"w{w}_fixed" in S and f"w{w}_low_margin" in S]
    if not pairs:
        print("   NOT GRADED — no matched-weight pair in the grid.")
    else:
        ok3 = True
        for w, lm, fx in pairs:
            d = S[lm][key] - S[fx][key]
            ok3 &= d > 0
            print(f"   w={w:<5} low_margin {S[lm][key]:.4f}  vs fixed {S[fx][key]:.4f}"
                  f"   {d:+.4f}  {'low_margin' if d > 0 else 'FIXED'}")
        print(f"   ->  {'PASS' if ok3 else 'FAIL'}")
        if not ok3:
            print("   The gating is complexity for nothing — report `fixed` as the honest")
            print("   choice rather than keeping a mode that does not earn itself.")

print("\n4. the best arm beats the control by more than 0.001")
if not EXACT:
    print("   NOT GRADED — score column is NaN.")
else:
    cands = [(S[a][key], a) for a in ARMS if a != CTRL and a in S]
    bv, ba = max(cands)
    d = bv - S[CTRL][key]
    ok4 = d > 0.001
    for v, a in sorted(cands, reverse=True):
        print(f"   {a:<18} {v:.4f}  ({v-S[CTRL][key]:+.4f})")
    print(f"   best {ba} at {d:+.4f}  ->  {'PASS' if ok4 else 'FAIL'}")
    if not ok4:
        print("   The second model adds nothing on our pipeline. That closes the LAST")
        print("   identified lever (notes/40 §4), and the honest conclusion is that the")
        print("   0.926 field's advantage is not reproducible from what is public —")
        print("   0.899-0.903 is where this approach lands.")

print("\n5. the optimum is not at the largest weight tried")
if not EXACT:
    print("   NOT GRADED — score column is NaN.")
else:
    ws = sorted({w for w, m in GRID if w > 0})
    vals = [(w, max(S[a][key] for a in ARMS
                    if a.startswith("w" + str(w) + "_") and a in S)) for w in ws]
    if len(vals) >= 3:
        bw = max(vals, key=lambda kv: kv[1])[0]
        ok5 = bw != ws[-1]
        print("   " + "  ".join(f"w{w}:{v:.4f}" for w, v in vals))
        print(f"   best w={bw}  ->  {'PASS — interior or at the floor' if ok5 else 'FAIL — still climbing'}")
    else:
        print(f"   NOT GRADED — {len(vals)} weights.")

print()
print("=" * 88)
if EXACT:
    best = max(ARMS, key=lambda a: S[a][key] if a in S else float("-inf"))
    d = S[best][key] - S[CTRL][key]
    print(f"BEST ARM: {best} at {S[best][key]:.4f}  ({d:+.4f} vs control)")
    print(f"  mislink {A[best]['fn_mislink']:,}  gap {A[best]['fn_gap']:,}  "
          f"detect {A[best]['fn_detect']:,}  div_J {S[best].get('division_jaccard', 0):.4f}")
    if d <= 0.001:
        print("  INSIDE NOISE. The last identified lever does not pay. Every direction from")
        print("  notes/33's audit is now measured, and the remaining gap to 0.926 is not")
        print("  explained by anything in the public configuration or the public weights.")
    else:
        print(f"  Submittable: secondary at {best}.")
else:
    print("NO BEST ARM — score column is NaN.")
print("=" * 88)